# PathSnap Model Training & Benchmark: Logistic Regression (Linear & Probabilistic)
**Course:** UE24CS352A - Machine Learning (PES University)  
**Team 6:** PathSnap Distributed Map-Matching  
**Model Name:** `logistic`  

---
### Notebook Objectives
1. Inspect hyperparameters, training pipeline, and environment provenance.
2. Review 5-fold group-aware cross-validation on 80 training trips (16 trips/fold).
3. Evaluate held-out generalization on the 10 student-test trips.
4. Verify observation-level Top-1 accuracy, Macro F1, and categorical log loss.


In [1]:
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, HTML, Image

PROJ_ROOT = Path("..").resolve()
MODEL_DIR = PROJ_ROOT / "results" / "run_01" / "logistic"
sys.path.insert(0, str(PROJ_ROOT / "scripts"))

print("Model output directory:", MODEL_DIR)


Model output directory: C:\Users\joshwin\PESU\3RD YEAR\5th sem\ML\team_6\results\run_01\logistic


## 1. Model Configuration & Provenance
Inspection of hyperparameters, random seeds, and package dependencies.


In [2]:
with open(MODEL_DIR / "configuration.json") as f:
    config = json.load(f)
with open(MODEL_DIR / "provenance.json") as f:
    prov = json.load(f)

print(f"Algorithm: {prov['model']} (Random Seed: {prov['seed']})")
print(f"Scikit-Learn Version: {prov['sklearn']} | Python: {prov['python']}")
print("\nKey Parameters:")
for k in sorted(config.keys())[:12]:
    print(f"  {k}: {config[k]}")


Algorithm: logistic (Random Seed: 212)
Scikit-Learn Version: 1.9.1 | Python: 3.12.6

Key Parameters:
  logisticregression: LogisticRegression(max_iter=1000, random_state=212)
  logisticregression__C: 1.0
  logisticregression__class_weight: None
  logisticregression__dual: False
  logisticregression__fit_intercept: True
  logisticregression__intercept_scaling: 1
  logisticregression__l1_ratio: 0.0
  logisticregression__max_iter: 1000
  logisticregression__n_jobs: None
  logisticregression__penalty: deprecated
  logisticregression__random_state: 212
  logisticregression__solver: lbfgs


## 2. 5-Fold Cross-Validation & Student-Test Evaluation
Group-aware 5-fold evaluation across all 80 training trips, followed by refitting on the full training set and testing on the 10 student-test trips.


In [3]:
with open(MODEL_DIR / "metrics.json") as f:
    records = json.load(f)

rows = []
for r in records:
    m = r["metrics"]["all_points"]
    rel = r["metrics"]["reliable_points"]
    rows.append({
        "Split": r["split"],
        "Top-1 Accuracy": f"{m['top1_accuracy']*100:.2f}%",
        "Macro F1": f"{m['macro_f1']:.4f}",
        "Macro Precision": f"{m['macro_precision']:.4f}",
        "Macro Recall": f"{m['macro_recall']:.4f}",
        "Log Loss": f"{m['categorical_log_loss_with_miss_state']:.4f}",
        "Reliable Top-1": f"{rel['top1_accuracy']*100:.2f}%",
        "Fit Time (s)": f"{r['fit_seconds']:.1f}s",
        "Inf Time (ms)": f"{r['inference_seconds']*1000:.1f}ms"
    })

df_metrics = pd.DataFrame(rows)
display(df_metrics)


,Split,Top-1 Accuracy,Macro F1,Macro Precision,Macro Recall,Log Loss,Reliable Top-1,Fit Time (s),Inf Time (ms)
0,fold_0,77.76%,0.6129,0.6513,0.5958,1.0660,85.86%,2.1s,16.4ms
1,fold_1,74.66%,0.6765,0.7093,0.6617,3.3818,86.50%,1.6s,24.2ms
2,fold_2,46.03%,0.5669,0.6129,0.5452,9.8203,79.71%,2.3s,27.4ms
3,fold_3,75.57%,0.6525,0.6914,0.6325,0.8608,77.88%,2.3s,54.3ms
4,fold_4,85.09%,0.7055,0.7366,0.6924,0.7036,90.79%,2.1s,14.3ms
5,student_test,80.76%,0.6284,0.6711,0.6113,0.7729,88.16%,2.7s,10.0ms


## 3. Performance Summary
Mean cross-validation metrics across all 5 folds.


In [4]:
with open(MODEL_DIR / "summary.json") as f:
    summary = json.load(f)

print(f"Completed Folds: {summary['completed_folds']}")
print(f"Mean CV Macro F1: {summary['cv_macro_f1_mean']:.4f} ± {summary['cv_macro_f1_std']:.4f}")
print(f"Mean CV Top-1 Accuracy: {summary['cv_top1_acc_mean']*100:.2f}%")
print(f"Student-Test Evaluation Complete: {summary['student_test_completed']}")


Completed Folds: 5
Mean CV Macro F1: 0.6428 ± 0.0486
Mean CV Top-1 Accuracy: 71.82%
Student-Test Evaluation Complete: True
